# 07 — Silver Payment Status CDC

**Input:** `Capstone_team01.bronze.payment_status_changes`  
**Outputs:** `Capstone_team01.silver.payment_status_changes` and its quarantine table.

Run `08_silver_master_data` and `05_silver_transactions` first.

Rules:
- `change_event_id` is required and unique.
- `change_ts` must parse as a timestamp when present.
- `change_sequence` must parse as an integer when present.
- `transaction_id` must exist in Silver transactions.
- Duplicate change event IDs are quarantined in full.
- This notebook preserves every valid change event; it does not collapse events into a latest authorization state. That state-building task belongs to Day 3.


In [ ]:
from pyspark.sql import functions
from pyspark.sql.window import Window

spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("Capstone_team01.bronze.payment_status_changes")

df = (df
      .withColumn("_raw_change_ts", functions.col("change_ts"))
      .withColumn("_raw_change_sequence", functions.col("change_sequence"))
      .withColumn("change_ts", functions.to_timestamp("change_ts"))
      .withColumn("change_sequence", functions.col("change_sequence").cast("integer")))

duplicate_window = Window.partitionBy("change_event_id")
df = df.withColumn("_key_count", functions.count("*").over(duplicate_window))

df = df.withColumn(
    "_dq_reason",
    functions.concat_ws(
        ";",
        functions.when(
            functions.col("change_event_id").isNull() |
            (functions.trim(functions.col("change_event_id")) == ""),
            "MISSING_CHANGE_EVENT_ID"
        ),
        functions.when(functions.col("_key_count") > 1, "DUPLICATE_CHANGE_EVENT_ID"),
        functions.when(functions.col("_raw_change_ts").isNotNull() & functions.col("change_ts").isNull(), "INVALID_CHANGE_TS"),
        functions.when(functions.col("_raw_change_sequence").isNotNull() & functions.col("change_sequence").isNull(), "INVALID_CHANGE_SEQUENCE")
    )
)

transactions = spark.table("Capstone_team01.silver.card_transactions").select("transaction_id").distinct()
df = df.join(
    transactions.withColumn("_transaction_found", functions.lit(True)),
    "transaction_id",
    "left"
)

df = df.withColumn(
    "_dq_reason",
    functions.concat_ws(
        ";", functions.col("_dq_reason"),
        functions.when(functions.col("_transaction_found").isNull(), "UNKNOWN_TRANSACTION_ID")
    )
).withColumn("_quarantined_at", functions.current_timestamp())

quarantine_df = df.filter(functions.col("_dq_reason") != "")
clean_df = df.filter(functions.col("_dq_reason") == "")

clean_df = clean_df.drop(
    "_raw_change_ts", "_raw_change_sequence", "_key_count",
    "_dq_reason", "_quarantined_at", "_transaction_found"
)

(
    clean_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.payment_status_changes")
)
(
    quarantine_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.quarantine_payment_status_changes")
)

print("Clean payment changes:", clean_df.count())
print("Quarantined payment changes:", quarantine_df.count())
display(clean_df.limit(10))
